In [1]:
#Data Cleaning and Preprocessing for UK Lung Cancer Mortality Model
# This script prepares the data for a Bayesian model of lung cancer mortality in the UK.
# It loads mortality and population data, processes it, and prepares it for modeling.
# Import necessary libraries
import os
import numpy as np
import pandas as pd 
import geopandas as gpd 
from jax.nn import sigmoid

import numpyro
numpyro.set_host_device_count(4)
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive
from numpyro.diagnostics import hpdi

import jax
import jax.numpy as jnp

import arviz as az
import matplotlib.pyplot as plt
from libpysal.weights import Queen
# Check number of devices available
num_device = jax.local_device_count()
print("Number of available devices: ", num_device)

#Data Cleaning Overview: 
# Filter data to only English regions 
# and remove unnecessary columns.
#Merge mortality and population data on LAD_code.
#Merge sjapefile data with mortality and population data.


# Loading aggrgated mortality data 
# Loading mortality data 
data = pd.read_csv("/Users/alydiaowens/uk-lung-model/ukaggregatedlung.csv", skiprows=8)  # skipping metadata
data = data.rename(columns={
    "mnemonic": "LAD_code",
    "2023": "Lung Cancer Deaths"
})
print("Mortality Data", data.head(13))

#loading population data 
pop_data = pd.read_csv("/Users/alydiaowens/uk-lung-model/ukaggregatedpop.csv", skiprows=8)  # skipping metadata
pop_data = pop_data.rename(columns={
    "mnemonic": "LAD_code",
    "2023": "Population"
})
print("Population Data", pop_data.head(13))

#loading shapefile 
# Loading geoportal data
gdf = gpd.read_file("/Users/alydiaowens/uk-lung-model/Regions-Dec23/Regions-Dec2023/RGN_DEC_2023_EN_BUC.shp")
print("GDF Data", gdf.head(13)) 



/Users/alydiaowens/uk-lung-model/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/alydiaowens/uk-lung-model/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Number of available devices:  4
Mortality Data                       region      LAD_code  Lung Cancer Deaths
0                 North East     E12000001                1808
1                 North West     E12000002                4236
2   Yorkshire and The Humber     E12000003                3022
3              East Midlands     E12000004                2498
4              West Midlands     E12000005                2865
5                       East     E12000006                2679
6                     London     E12000007                2422
7                 South East     E12000008                3859
8                 South West     E12000009                2663
9                      Wales     W92000004                1816
10         Unknown or Abroad             Z                  55
11              Column Total  Column Total               27923
Population Data                   North East                   E12000001  2711380
0                 North West                   E1200